# SceneHawk - fine-tune the feelings model

Fine-tunes `distilroberta-base` with 28 independent sigmoid outputs, one per
feeling, on the hand- and LLM-labelled reviews. Run this in Colab on a T4 GPU.
It takes about 10 minutes.

**Upload** these four files from `chatbot/data/training/`. If they are missing,
regenerate them with `python scripts/09_build_training_data.py`:

| file | used for |
| --- | --- |
| `meta.json` | feeling order, plus counts that check the upload |
| `train.jsonl` | training |
| `val.jsonl` | picking the best epoch |
| `answer_key.jsonl` | predicted on, **never scored here** |

**Download**: `distilroberta_preds.zip`, which holds the val and answer-key
predictions and a run record.

**Saved to Google Drive**: the trained model, in
`MyDrive/scenehawk/models/<version>/`, with a copy of the zip. Colab asks for
Drive access right after the upload, so stay at the keyboard until training
starts. Without this step the model is lost when the runtime disconnects.

Two rules carry over from `eval/score.py`:

1. **No thresholds here.** The notebook writes probabilities.
   `score.py --tune` fits the cut points on val, locally, the same way it did
   for the baseline, so both systems are thresholded by the same procedure.
2. **The answer key is not looked at.** Its labels are dropped as it loads,
   and the notebook computes no metric on it. Picking an epoch by its
   answer-key score would turn the test set into a tuning set.

The number to beat is the tf-idf baseline: **0.157** macro-F1 over the 18
reportable feelings.

In [ ]:
# Colab already ships torch, transformers and scikit-learn; nothing to install.
import json
import os
import random
import re
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import sklearn
import torch
import transformers
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

print(f"torch {torch.__version__}, transformers {transformers.__version__}, "
      f"scikit-learn {sklearn.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    # On CPU this takes hours, and fp16 is off, so the run would not match a
    # GPU one anyway. Fail loudly instead of looking stuck.
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## Settings

Every knob is here, and all of them go into the run record that is downloaded
with the predictions.

- **`distilroberta-base`**: 82M parameters, which is about the most 2,643
  reviews can fine-tune without memorising them. RoBERTa's pretraining
  includes web text and reviews, which are close to IMDb prose.
- **`MAX_LEN = 512`**: the median training review is 219 tokens, and 12% run
  past 512 and get truncated.
- **`SEED = 49900`**: the same seed as sampling and the baseline. GPU kernels
  are not bit-for-bit deterministic, so a rerun can move the third decimal.

In [ ]:
MODEL = "distilroberta-base"
RUN = "distilroberta"          # prefix for the output files
SEED = 49900
MAX_LEN = 512
BATCH = 16
EPOCHS = 6
LR = 3e-5
WARMUP = 0.1                   # fraction of steps spent warming up
WEIGHT_DECAY = 0.01

# Goes in the run record and later in film_feelings.model_version, so every
# stored score can be traced to the weights that produced it. Each run gets its
# own folder, so a rerun can't overwrite weights the committed predictions
# came from.
MODEL_VERSION = f"{RUN}-{time.strftime('%Y%m%d-%H%M', time.gmtime())}"
print("model version:", MODEL_VERSION)

## Load the data

Colab saves a re-uploaded file as `train (1).jsonl`. The loader strips that
suffix, so uploading twice still works.

Run outside Colab, the notebook reads straight from `../data/training/` and
writes to `../eval/preds/`.

In [ ]:
NEED = ["meta.json", "train.jsonl", "val.jsonl", "answer_key.jsonl"]

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA = Path("/content/data")
    OUT = Path("/content/preds")
    DATA.mkdir(exist_ok=True)
    # Mounted now, not after training, so Run all can't finish and then sit on
    # a permission prompt with the only copy of the model in memory.
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = Path("/content/drive/MyDrive/scenehawk/models") / MODEL_VERSION
    missing = [n for n in NEED if not (DATA / n).exists()]
    if missing:
        print("Upload from chatbot/data/training/:", ", ".join(missing))
        for name, blob in colab_files.upload().items():
            (DATA / re.sub(r" \(\d+\)(?=\.)", "", name)).write_bytes(blob)
else:
    DATA = Path("../data/training")
    OUT = Path("../eval/preds")
    SAVE = Path("../models") / MODEL_VERSION

missing = [n for n in NEED if not (DATA / n).exists()]
assert not missing, f"still missing: {', '.join(missing)} - rerun this cell"
OUT.mkdir(parents=True, exist_ok=True)
SAVE.mkdir(parents=True, exist_ok=True)
print("data:", DATA.resolve())
print("model will be saved to:", SAVE)

In [ ]:
meta = json.loads((DATA / "meta.json").read_text(encoding="utf-8"))
FEELINGS = meta["feelings"]
assert len(FEELINGS) == 28 and len(set(FEELINGS)) == 28


def read(split):
    with open(DATA / f"{split}.jsonl", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


train = read("train")
val = read("val")
# The answer key's labels are thrown away on load, so nothing below can use
# them, even by accident. It is scored locally by eval/score.py.
answer_key = [{"id": r["id"], "imdb_id": r["imdb_id"], "text": r["text"]}
              for r in read("answer_key")]

# A stale or half-finished upload has to fail here, not show up as a
# strange score an hour later.
for name, rows in (("train", train), ("val", val), ("answer_key", answer_key)):
    expected = meta["splits"][name]["rows"]
    assert len(rows) == expected, f"{name}: {len(rows)} rows, meta.json says {expected}"
for name, rows in (("train", train), ("val", val)):
    assert all(len(r["labels"]) == 28 for r in rows), f"{name}: a label vector is not 28 long"

Y_train = np.array([r["labels"] for r in train], dtype=np.float32)
Y_val = np.array([r["labels"] for r in val], dtype=np.float32)
counted = dict(zip(FEELINGS, Y_train.sum(0).astype(int).tolist()))
assert counted == meta["train_positives"], "train.jsonl does not match meta.json - re-upload both"

# The splits are by film. If one film's reviews were on both sides, val would
# reward memorising that film instead of reading the review.
films = {name: {r["imdb_id"] for r in rows}
         for name, rows in (("train", train), ("val", val), ("answer_key", answer_key))}
assert not films["train"] & films["val"], "a film is in both train and val"
assert not films["train"] & films["answer_key"], "a film is in both train and answer_key"
assert not films["val"] & films["answer_key"], "a film is in both val and answer_key"

print(f"train {len(train):,} reviews, {int(Y_train.sum()):,} marks, {len(films['train']):,} films")
print(f"val {len(val):,} reviews, {int(Y_val.sum()):,} marks")
print(f"answer_key {len(answer_key):,} reviews (labels dropped)")
print("all checks passed")

## Class weights

`amused` has 507 training positives and `confused` has 6. With an unweighted
loss the cheapest answer for every rare feeling is "no", so each feeling's
positives are up-weighted in `BCEWithLogitsLoss`.

The weight is **√(negatives / positives)**, not the raw ratio. The raw ratio
gives `confused` a weight of ~440, so each of its 6 reviews would outweigh
whole batches, and the model would memorise those six. The square root gives
it ~21 and `amused` ~2. That still pushes hard toward rare feelings without
letting six reviews drive training.

`lonely` has no training positives and gets weight 1. It can only ever learn
"no". That's why it sits in the `cannot_learn` tier.

In [ ]:
pos = Y_train.sum(0)
neg = len(Y_train) - pos
pos_weight = np.where(pos > 0, np.sqrt(neg / np.maximum(pos, 1)), 1.0).astype(np.float32)

print(f"{'feeling':<16}{'train +':>8}{'val +':>7}{'weight':>8}")
for f, p, v, w in zip(FEELINGS, pos.astype(int), Y_val.sum(0).astype(int), pos_weight):
    print(f"{f:<16}{p:>8}{v:>7}{w:>8.1f}")

## Tokenise

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)


class Reviews(Dataset):
    def __init__(self, rows, labels=None):
        self.enc = tok([r["text"] for r in rows], truncation=True, max_length=MAX_LEN)
        self.labels = labels

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


def collate(batch):
    # Pad each batch to its own longest review rather than to 512.
    labels = [b.pop("labels") for b in batch] if "labels" in batch[0] else None
    out = tok.pad(batch, return_tensors="pt")
    if labels is not None:
        out["labels"] = torch.tensor(np.stack(labels))
    return out


train_ds = Reviews(train, Y_train)
val_ds = Reviews(val)
answer_ds = Reviews(answer_key)

lengths = np.array([len(x) for x in train_ds.enc["input_ids"]])
print(f"train tokens: median {int(np.median(lengths))}, "
      f"truncated at {MAX_LEN}: {(lengths >= MAX_LEN).mean():.0%} of reviews")

## Train

Each epoch ends with a val check, and the best epoch is kept. "Best" means
**mean average precision** over the feelings that have val positives. It's
threshold-free, which fits because thresholds are not chosen until
`score.py --tune`. Picking an epoch by F1 here would need thresholds, and
those would then be fitted twice.

For reference, the tf-idf baseline scores **0.224** on this same measure
(23 feelings). An epoch below that is losing to bag-of-words.

Loading the model prints a report listing `classifier.*` as **MISSING** and
`lm_head.*` as **UNEXPECTED**. That's expected. The 28-output head is new and
starts random, and the language-model head isn't used.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(net, ds):
    net.eval()
    probs = []
    for batch in DataLoader(ds, batch_size=BATCH * 2, collate_fn=collate):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = net(**batch).logits
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(probs)


def mean_ap(y, s):
    has = [i for i in range(y.shape[1]) if y[:, i].sum() > 0]
    return float(np.mean([average_precision_score(y[:, i], s[:, i]) for i in has])), len(has)


set_seed(SEED)
# The feeling names go in the model's config, so the saved model carries its
# own output order.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(FEELINGS), problem_type="multi_label_classification",
    id2label=dict(enumerate(FEELINGS)), label2id={f: i for i, f in enumerate(FEELINGS)},
).to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

# The usual split: no weight decay on biases and LayerNorm weights.
no_decay = ("bias", "LayerNorm.weight")
groups = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
     "weight_decay": 0.0},
]
opt = torch.optim.AdamW(groups, lr=LR)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))
steps = EPOCHS * len(train_dl)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

history, best = [], {"val_mean_ap": -1.0}
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, total = time.time(), 0.0
    for batch in train_dl:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item()

    val_probs = predict(model, val_ds)
    ap, n = mean_ap(Y_val, val_probs)
    history.append({"epoch": epoch, "train_loss": round(total / len(train_dl), 4),
                    "val_mean_ap": round(ap, 4)})
    mark = ""
    if ap > best["val_mean_ap"]:
        best = {"epoch": epoch, "val_mean_ap": ap}
        # Copied to CPU so the GPU copy keeps training without overwriting it.
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        mark = "  <- best so far"
    print(f"epoch {epoch}: train loss {total / len(train_dl):.4f}, "
          f"val mean AP {ap:.3f} ({n} feelings), {time.time() - t0:.0f}s{mark}")

model.load_state_dict(best_state)
print(f"\nkept epoch {best['epoch']}: val mean AP {best['val_mean_ap']:.3f} (tf-idf baseline 0.224)")

## Save the model

Saved before anything else, because it's the one thing that can't be recovered
without training again. The saved copy is then loaded back and has to reproduce
the kept model's val scores, so a bad save stops here rather than surfacing when
the feelings table gets filled.

The folder holds the weights, the tokenizer and `meta.json`, which together are
everything needed to score new reviews.

In [ ]:
model.save_pretrained(SAVE)
tok.save_pretrained(SAVE)
shutil.copy(DATA / "meta.json", SAVE / "meta.json")

kept_probs = predict(model, val_ds)
kept_ap, _ = mean_ap(Y_val, kept_probs)
assert abs(kept_ap - best["val_mean_ap"]) < 1e-3, "the model in memory is not the best epoch"

saved = AutoModelForSequenceClassification.from_pretrained(str(SAVE)).to(DEVICE)
assert [saved.config.id2label[i] for i in range(len(FEELINGS))] == FEELINGS, "saved feeling order is wrong"
assert np.allclose(predict(saved, val_ds), kept_probs, atol=1e-3), "the saved model scores differently"
del saved

size = sum(p.stat().st_size for p in SAVE.iterdir()) / 1e6
print(f"saved and checked: {SAVE} ({size:.0f} MB), epoch {best['epoch']}")

## Write the predictions

The format is the same as `scripts/10_baseline.py`'s: one
`{"id": ..., "scores": [28 probabilities in meta.json order]}` per line.
`score.py` reads either file the same way.

In [ ]:
def write(rows, probs, path):
    with open(path, "w", encoding="utf-8") as fh:
        for r, p in zip(rows, probs):
            fh.write(json.dumps({"id": r["id"], "scores": [round(float(v), 5) for v in p]}) + "\n")


val_probs = predict(model, val_ds)
answer_probs = predict(model, answer_ds)
assert val_probs.shape == (len(val), 28) and answer_probs.shape == (len(answer_key), 28)

files = {
    f"{RUN}_val.jsonl": (val, val_probs),
    f"{RUN}_answer_key.jsonl": (answer_key, answer_probs),
}
for name, (rows, probs) in files.items():
    write(rows, probs, OUT / name)

run = {
    "model_version": MODEL_VERSION, "weights": str(SAVE),
    "model": MODEL, "seed": SEED, "max_len": MAX_LEN, "batch": BATCH,
    "epochs": EPOCHS, "lr": LR, "warmup": WARMUP, "weight_decay": WEIGHT_DECAY,
    "pos_weight": "sqrt(neg/pos), 1 where a feeling has no train positives",
    "selection": "best epoch by mean average precision on val",
    "best_epoch": best["epoch"], "val_mean_ap": round(best["val_mean_ap"], 4),
    "history": history,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "scikit-learn": sklearn.__version__},
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
}
(OUT / f"{RUN}_run.json").write_text(json.dumps(run, indent=2) + "\n", encoding="utf-8")

if IN_COLAB:
    bundle = Path(f"/content/{RUN}_preds.zip")
    with zipfile.ZipFile(bundle, "w") as z:
        for name in [*files, f"{RUN}_run.json"]:
            z.write(OUT / name, name)
    shutil.copy(bundle, SAVE / bundle.name)
    shutil.copy(OUT / f"{RUN}_run.json", SAVE / f"{RUN}_run.json")
    colab_files.download(str(bundle))
print("wrote", ", ".join([*files, f"{RUN}_run.json"]), "to", OUT.resolve())

if IN_COLAB:
    # Drive writes are buffered. This waits until they have really uploaded.
    drive.flush_and_unmount()
    print(f"Google Drive: MyDrive/scenehawk/models/{MODEL_VERSION}/")
    print("If the zip did not download, there is a copy in that folder.")

## Then, back on your machine

Unzip into `chatbot/eval/preds/` and move `distilroberta_run.json` up to
`chatbot/eval/`. Then, from `chatbot/`:

```
python eval/score.py eval/preds/distilroberta_val.jsonl --tune --out eval/thresholds_distilroberta.json
python eval/score.py eval/preds/distilroberta_answer_key.jsonl --thresholds eval/thresholds_distilroberta.json
```